In [1]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import cross_val_score
import joblib
from model_wrapper import WrappedXGBModel
import sqlite3

In [2]:
# first things first, extract tables from database
# conn = sqlite3.connect("../backend/database/reports.db")

# actually first, extract DB from test CSV file:

In [3]:
master_df = pd.read_csv("../machine-learning-model-prep/data/mock_training_dataset_train.csv")
test_df = pd.read_csv("../machine-learning-model-prep/data/mock_training_dataset_test.csv")
master_df.head()
master_df.dtypes

report_id                       int64
community_name                    str
region_id                       int64
report_date                       str
reporter_type                     str
local_season                      str
rainfall_level                    str
road_access                       str
seasonal_indicators_text          str
num_dogs_seen                   int64
num_puppies_seen                int64
skin_issue_count                int64
parasite_issue_count            int64
recent_dog_deaths               int64
distance_to_clinic              int64
dog_roaming_level                 str
requested_help                  int64
notes                             str
months_since_last_visit         int64
needs_parasite_treatment_3m     int64
needs_scabies_treatment_3m      int64
needs_followup_visit_3m         int64
needs_parasite_treatment_6m     int64
needs_scabies_treatment_6m      int64
needs_followup_visit_6m         int64
needs_parasite_treatment_9m     int64
needs_scabie

In [4]:
# so first removing un-needed columns:
print(master_df.columns.tolist())
master_df = master_df.drop(columns=["community_name", "report_id", "reporter_type", "report_date", "seasonal_indicators_text", "notes"])
test_df = test_df.drop(columns=["community_name", "report_id", "reporter_type", "report_date", "seasonal_indicators_text", "notes"])

['report_id', 'community_name', 'region_id', 'report_date', 'reporter_type', 'local_season', 'rainfall_level', 'road_access', 'seasonal_indicators_text', 'num_dogs_seen', 'num_puppies_seen', 'skin_issue_count', 'parasite_issue_count', 'recent_dog_deaths', 'distance_to_clinic', 'dog_roaming_level', 'requested_help', 'notes', 'months_since_last_visit', 'needs_parasite_treatment_3m', 'needs_scabies_treatment_3m', 'needs_followup_visit_3m', 'needs_parasite_treatment_6m', 'needs_scabies_treatment_6m', 'needs_followup_visit_6m', 'needs_parasite_treatment_9m', 'needs_scabies_treatment_9m', 'needs_followup_visit_9m', 'needs_parasite_treatment_12m', 'needs_scabies_treatment_12m', 'needs_followup_visit_12m']


In [5]:
# 1. choose columns
input_cols = master_df.loc[:, "region_id":"requested_help"].columns.tolist()
target_cols = master_df.loc[:, "needs_parasite_treatment_3m":"needs_followup_visit_12m"].columns.tolist()

# 2. split X and Y
X_train = master_df[input_cols].copy()
Y_train = master_df[target_cols].copy()

X_test = test_df[input_cols].copy()
Y_test = test_df[target_cols].copy()

# 3. convert categorical inputs to pandas category dtype
cat_cols = ["region_id", "local_season", "rainfall_level", "road_access", "dog_roaming_level", "requested_help"]
for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

# 4. base XGBoost model
base_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    enable_categorical=True,
    random_state=42
)

# 5. wrap XGBoost for multiple output columns
model = MultiOutputClassifier(base_model)

# OK SO HEre.... I NEED A TRAIN/TEST TING LOLLLLLLLLLL HAHAHAHAHAAHHAAHAHAAAHAHAAHAHA

# 6. train
model.fit(X_train, Y_train)

predictions = model.predict(X_test)

prediction_df = pd.DataFrame(predictions, columns=target_cols, index=Y_test.index)

prediction_df.head()

,needs_parasite_treatment_3m,needs_scabies_treatment_3m,needs_followup_visit_3m,needs_parasite_treatment_6m,needs_scabies_treatment_6m,needs_followup_visit_6m,needs_parasite_treatment_9m,needs_scabies_treatment_9m,needs_followup_visit_9m,needs_parasite_treatment_12m,needs_scabies_treatment_12m,needs_followup_visit_12m
0,1,1,1,1,0,1,0,1,1,0,1,1
1,1,1,1,1,1,1,0,1,1,0,0,1
2,1,1,0,1,1,1,1,1,1,1,1,0
3,1,0,1,1,0,1,0,0,0,0,1,1
4,1,1,1,1,0,0,0,0,1,1,1,1


In [6]:
# precision measures how often the model is right when it predicts '1'
# recall measures of all the actual 1s, how many did the model find
# f1 is an evaluation metric that is a combination of both precision and recall scores.

# below is cross-validation

for col in Y_train.columns:
    scores = cross_val_score(
        base_model,
        X_train,
        Y_train[col],
        cv=5,
        scoring="f1"
    )

    print(col)
    print("Scores:", scores)
    print("Mean F1:", scores.mean())

needs_parasite_treatment_3m
Scores: [0.76521739 0.81034483 0.77227723 0.78947368 0.81081081]
Mean F1: 0.789624788326933
needs_scabies_treatment_3m
Scores: [0.56338028 0.58823529 0.51428571 0.54320988 0.56410256]
Mean F1: 0.5546427461478552
needs_followup_visit_3m
Scores: [0.65306122 0.64646465 0.74747475 0.64       0.66666667]
Mean F1: 0.6707334570191713
needs_parasite_treatment_6m
Scores: [0.68888889 0.64646465 0.51219512 0.62921348 0.61904762]
Mean F1: 0.6191619518996883
needs_scabies_treatment_6m
Scores: [0.43835616 0.36619718 0.28       0.36619718 0.60273973]
Mean F1: 0.4106980513216284
needs_followup_visit_6m
Scores: [0.58426966 0.57471264 0.46341463 0.65168539 0.65909091]
Mean F1: 0.5866346486190374
needs_parasite_treatment_9m
Scores: [0.52054795 0.50666667 0.49275362 0.58974359 0.55072464]
Mean F1: 0.5320872924970603
needs_scabies_treatment_9m
Scores: [0.48648649 0.41176471 0.47457627 0.3880597  0.33846154]
Mean F1: 0.4198697407018712
needs_followup_visit_9m
Scores: [0.60674157 

In [7]:
# Here is evaluation on the testing dataset
for col in Y_test.columns:
    print(f"\nTarget: {col}")
    print(classification_report(Y_test[col], prediction_df[col]))


Target: needs_parasite_treatment_3m
              precision    recall  f1-score   support

           0       0.53      0.30      0.38        33
           1       0.68      0.84      0.75        58

    accuracy                           0.65        91
   macro avg       0.60      0.57      0.57        91
weighted avg       0.62      0.65      0.62        91


Target: needs_scabies_treatment_3m
              precision    recall  f1-score   support

           0       0.56      0.40      0.46        48
           1       0.49      0.65      0.56        43

    accuracy                           0.52        91
   macro avg       0.53      0.52      0.51        91
weighted avg       0.53      0.52      0.51        91


Target: needs_followup_visit_3m
              precision    recall  f1-score   support

           0       0.38      0.32      0.35        25
           1       0.76      0.80      0.78        66

    accuracy                           0.67        91
   macro avg       0.5

In [19]:
# Now I want to use the class ting
drop_cols = [
    "community_name"
    "report_id",
    "reporter_type",
    "report_date",
    "seasonal_indicators_text",
    "notes"
]

wrapped = WrappedXGBModel(
    input_cols=input_cols,
    cat_cols=cat_cols,
    target_cols=target_cols,
    drop_cols=drop_cols
)

wrapped.fit(master_df, Y_train)

new_prediction_df = wrapped.predict(test_df)
new_prediction_df.head()

joblib.dump(wrapped, "wrapped_XGB_model.joblib")

['wrapped_XGB_model.joblib']

In [20]:
# The following is how to use the wrapped model:

In [ ]:
# USE THE BELOW IN A PYTHON FILE:
# BUT YOU NEED 2 FILES IN YOUR SAME DIRECTORY: model_wrapper.py AND wrapped_XGB_model.joblib
# BOTH OF THOSE ARE HERE IN THE SAME DIRECTORY AS THIS JUPYTER NOTEBOOK FILE

# import joblib
# from model_wrapper import WrappedXGBModel

# model = joblib.load("wrapped_model.joblib")
# predicted_df = model.predict(--YOUR DATAFRAME--)